Building a GPT From Scratch

**Prerequisites:** You know the Transformer architecture from lectures. Today you implement it.

You will build a **character-level decoder-only GPT** and train it on Shakespeare. The model is trained for **next-character prediction**: given a sequence of characters, it learns to predict what character comes next. This is the same self-supervised objective used by GPT-2/3/4, just at the character level instead of sub-word tokens. After training, we can generate new text by repeatedly sampling the next character from the model's predictions.

Most code is provided — your job is to **fill in the missing pieces** (marked `# TODO`, always 1–2 lines). Solutions are at the bottom.

**Design choices in this model:**
- **Token embeddings:** Learned lookup table via `nn.Embedding(vocab_size, n_embd)` — each character maps to a trainable vector. No pre-training (Word2Vec), no sub-word tokenization (Byte-Pair Encoding). With only 65 characters in the vocabulary, a simple lookup is sufficient.
- **Positional encoding:** Fixed sinusoidal encoding from *Attention Is All You Need* paper. Unlike learned positional embeddings, sinusoidal encodings can in principle generalize to sequence lengths unseen during training and add no trainable parameters.

Based on Andrej Karpathy's *Zero to Hero* series.

---
## 1. Data Preparation

Run **one** of the cells below depending on your OS to download the Tiny Shakespeare dataset.


In [ ]:
# === Linux / Google Colab ===
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [ ]:
# === macOS ===
!curl -sO https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [ ]:
# === Windows (PowerShell) ===
!powershell -Command "Invoke-WebRequest -Uri 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' -OutFile 'input.txt'"


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
torch.manual_seed(1337)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'{len(text):,} chars, vocab size {vocab_size}, train {len(train_data):,}, val {len(val_data):,}')


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

---
## 2. Hyperparameters & Helpers


In [ ]:
batch_size = 16       # how many independent sequences are processed in parallel per training step
block_size = 32       # maximum context length (number of characters the model sees at once)
max_iters = 5000      # total number of training steps
eval_interval = 100   # evaluate train/val loss every this many steps
learning_rate = 1e-3  # step size for the AdamW optimizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # use GPU if available, otherwise CPU
eval_iters = 200      # number of batches to average over when estimating loss
n_embd = 64           # embedding dimension (size of each token's vector representation)
n_head = 4            # number of parallel attention heads in each Transformer block
n_layer = 4           # number of stacked Transformer blocks
dropout = 0.0         # dropout rate (0 = no dropout; increase for regularization)

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print(f'Using device: {device}')

---
## 3. Self-Attention Intuition

Each token produces a **query** ("what am I looking for?"), a **key** ("what do I contain?"), and a **value** ("what do I communicate?"). Attention scores are the scaled dot product of queries and keys; the output is a weighted sum of values. A causal mask ensures tokens only attend to the past.


In [ ]:
# Quick demo of one self-attention head
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k, q = key(x), query(x)                            # (B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5    # (B, T, T) — scaled dot product

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))     # causal mask (so that our model cannot look into the future)
wei = F.softmax(wei, dim=-1)

out = wei @ value(x)                                # (B, T, head_size)
print(f'Output shape: {out.shape}')
print(f'Attention weights (row = query token, col = key token):\n{wei[0].detach()}')


---
## 4. Building the GPT — Exercises

Fill in each `# TODO` (1–2 lines). Hints are provided.


### A note on `register_buffer`

You will see `self.register_buffer('name', tensor)` in several places below. This is a PyTorch concept worth understanding:

- Some tensors are **not** learnable parameters (they should not be updated by the optimizer), but we still want them to *live inside the module* — travel with it when we call `.to(device)`, get saved/loaded with `state_dict()`, etc.
- `register_buffer` does exactly that: it stores the tensor on the module without adding it to `model.parameters()`.
- We use it for two things in this notebook:
  - **`tril`** — the lower-triangular causal mask in each attention head (a constant matrix of ones).
  - **`pe`** — the sinusoidal positional encoding matrix (pre-computed, never updated).

Think of it as: *"this tensor belongs to the model but is not trained."*


### Exercise 1: Single Attention Head

Compute the scaled dot-product attention scores:  $\;\text{wei} = QK^T / \sqrt{d_k}$


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # TODO: Compute scaled dot-product attention scores → (B, T, T)
        wei = ...  # <-- your code here

        # TODO: Apply the causal mask and softmax to the attention scores.
        #   1. Mask future positions with -inf:  wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        #   2. Apply softmax over correct dimension
        wei = ...  # <-- your code here (mask)
        wei = ...  # <-- your code here (softmax)

        wei = self.dropout(wei)
        return wei @ v

### Exercise 2: Multi-Head Attention

Run all heads in parallel and **concatenate** their outputs along the last dimension.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: Concatenate outputs of all heads along dim=-1
        # Hint: use torch.cat() with correct dim parameter applied on the list of head outputs [h(x) for h in self.heads]
        out = ...  # <-- your code here

        return self.dropout(self.proj(out))


### Exercise 3: Feed-Forward Network

Two linear layers with ReLU: `Linear(n_embd → 4*n_embd) → ReLU → Linear(4*n_embd → n_embd) → Dropout`.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # TODO: Define two linear layers for the feed-forward network.
        #   self.fc1 — maps from n_embd to 4 * n_embd (expansion)
        #   self.fc2 — maps from 4 * n_embd back to n_embd (projection)
        self.fc1 = ...      # <-- your code here
        self.fc2 = ...      # <-- your code here
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: Apply fc1, then ReLU, then fc2, then dropout.
        return ...  # <-- your code here

### Exercise 4: Transformer Block (Residual Connections)

Apply pre-norm residual connections: `x = x + sublayer(LayerNorm(x))`


In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        # TODO: Self-attention with residual connection
        x = ...  # <-- your code here
        # TODO: Feed-forward with residual connection
        x = ...  # <-- your code here
        return x


### Exercise 5: Sinusoidal Positional Encoding

The original Transformer uses fixed (non-learned) sinusoidal positional encodings. For a position $\text{pos}$ and embedding dimension index $i$:

$$PE_{(\text{pos}, 2i)} = \sin\!\left(\frac{\text{pos}}{10000^{2i/d}}\right), \qquad PE_{(\text{pos}, 2i+1)} = \cos\!\left(\frac{\text{pos}}{10000^{2i/d}}\right)$$

where $d$ = `n_embd`. We pre-compute a `(block_size, n_embd)` matrix and register it as a buffer (no gradients).

**Your task:** Fill in the two lines that compute the `sin` and `cos` values.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_embd, max_len=block_size):
        super().__init__()
        pe = torch.zeros(max_len, n_embd)            # (max_len, n_embd)
        position = torch.arange(max_len).unsqueeze(1) # (max_len, 1)
        # Shared denominator term: exp(-2i/d * ln(10000)) = 1 / 10000^(2i/d)
        div_term = torch.exp(torch.arange(0, n_embd, 2) * (-math.log(10000.0) / n_embd))

        # TODO: Fill the even columns (0, 2, 4, ...) with sin(position * div_term)
        pe[:, 0::2] = ...  # <-- your code here
        # TODO: Fill the odd columns (1, 3, 5, ...) with cos(position * div_term)
        pe[:, 1::2] = ...  # <-- your code here

        self.register_buffer('pe', pe)  # not a learnable parameter

    def forward(self, x):
        """x: (B, T, n_embd). Adds positional encoding for the first T positions."""
        return x + self.pe[:x.size(1)]  # (T, n_embd) broadcasts over batch


### Exercise 6: GPT Model — Putting It All Together

**Your task:** In `forward`, apply the positional encoding to the token embeddings by calling `self.pos_enc(tok_emb)`. This adds the sinusoidal vectors to the token representations before they enter the Transformer blocks.


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.pos_enc = SinusoidalPositionalEncoding(n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B, T, n_embd)
        # TODO: Apply sinusoidal positional encoding to tok_emb
        # Hint: self.pos_enc(tok_emb)
        x = ...  # <-- your code here

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


---
## 5. Training & Generation


In [ ]:
model = GPTLanguageModel().to(device)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f'step {iter:>5d}: train {losses["train"]:.4f}, val {losses["val"]:.4f}')
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


In [ ]:
# Generate Shakespeare!
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
